# TravelTide — Customer Segmentation & RFM Analysis
**Author:** Niharika Soni  
**Dataset window:** March–August 2023  
**Goal:** Segment 5,998 customers using RFM scoring and recommend personalised loyalty perks to improve retention.

---

> **Note on corrections:** This is the clean version of the analysis. Three issues found in the original notebook output have been fixed and documented in `CORRECTIONS.md`. Segment assignments were not affected by any of the fixes.

---

## Notebook Structure
1. [Setup & Data Load](#1-setup--data-load)
2. [EDA — User Behaviour](#2-eda--user-behaviour)
3. [EDA — Session Behaviour](#3-eda--session-behaviour)
4. [RFM Feature Engineering](#4-rfm-feature-engineering)
5. [RFM Scoring](#5-rfm-scoring)
6. [Segment Assignment](#6-segment-assignment)
7. [Segment Analysis](#7-segment-analysis)
8. [Perk Recommendations](#8-perk-recommendations)
9. [Export](#9-export)

## 1. Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Plotting defaults
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

print("Libraries loaded.")

### Data source
`customers.csv` is the aggregated user feature table produced by joining the raw `users` and `sessions` tables in Databricks SQL (see `sql/tables_filtering.sql`). One row per customer.

**Columns:**
| Column | Description |
|---|---|
| `user_id` | Unique customer identifier |
| `num_clicks` | Total page clicks across all sessions |
| `num_sessions` | Number of sessions (range: 8–12 in this dataset) |
| `avg_session_duration` | Mean session length in minutes |
| `avg_bags` | Average checked bags per booking |
| `age` | Customer age |
| `gender` | F / M / O |
| `married` | Boolean |
| `has_children` | Boolean |
| `home_country` | usa or canada |
| `home_city` / `home_airport` | Home location |
| `num_trips` | Total completed trips (our **Frequency** metric) |
| `num_flights` | Total flights booked |
| `money_spent_hotel` | Total hotel spend in USD (our **Monetary** metric) |
| `time_after_booking` | Days between last booking and last session (our **Recency** proxy) |
| `avg_km_flown` | Average km per flight |

In [ ]:
# Load data
df = pd.read_csv('data/raw/customers_ORIGINAL.csv')

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print(f"\nNull counts:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() or "  None — clean dataset")
print(f"\nDuplicate user_ids: {df['user_id'].duplicated().sum()}")
df.head(3)

---
## 2. EDA — User Behaviour

We start by understanding who our customers are: demographics, geography, and booking patterns.

In [ ]:
# ── Age distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['age'], bins=30, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Age Distribution')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Customers')
axes[0].axvline(df['age'].median(), color='tomato', linestyle='--', label=f'Median: {df["age"].median():.0f}')
axes[0].legend()

# Trip frequency
trip_counts = df['num_trips'].value_counts().sort_index()
axes[1].bar(trip_counts.index, trip_counts.values, color='steelblue', edgecolor='white')
axes[1].set_title('Trip Frequency Distribution')
axes[1].set_xlabel('Number of Trips')
axes[1].set_ylabel('Customers')
for i, v in zip(trip_counts.index, trip_counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

**Insight 1 — Middle-aged core audience.**  
Age peaks sharply between 35–55 (median: 43). Under-30 representation is thin. This is a mature, likely dual-income demographic. Pricing, UX, and perks should be calibrated accordingly — not for millennials or Gen Z.

**Insight 2 — Low-frequency bookers dominate.**  
Most customers have 2–4 trips. Very few power users (>6 trips). The retention lever is nudging 1–2 trip customers toward 3–4, not chasing outliers.

In [ ]:
# ── Demographics: gender, marital status, children ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, title in zip(axes, ['gender', 'married', 'has_children'],
                           ['Gender', 'Married', 'Has Children']):
    counts = df[col].value_counts()
    ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%',
           colors=['steelblue', 'lightcoral', 'lightgreen'], startangle=90)
    ax.set_title(title)

plt.tight_layout()
plt.show()

**Insight 3 — Predominantly female user base.**  
~88% of customers identify as female. This is a strong campaign signal — creatives are resonating with women. It also signals an untapped male audience (business travellers, solo adventurers) that could diversify revenue.

**Insight 4 — Mostly independent travellers.**  
67% have no children. Family-oriented perks (hotel add-ons for kids) have limited addressable market. Solo and couple travel deals have broader reach.

In [ ]:
# ── Geography: top home airports ─────────────────────────────────────────
top_airports = df['home_airport'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top_airports.index[::-1], top_airports.values[::-1], color='steelblue')
ax.set_title('Top 15 Home Airports by Customer Count')
ax.set_xlabel('Customers')
for i, v in enumerate(top_airports.values[::-1]):
    ax.text(v + 2, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

**Insight 5 — Concentrated in major US hubs.**  
Top airports are all major US hubs (JFK, LAX, ORD). Geo-targeted digital advertising to New York and Los Angeles will reach a disproportionate share of high-value users — more efficient than a broad national campaign.

In [ ]:
# ── Sign-up trend (using sign_up cohort from rfm output proxy) ───────────
# Note: sign_up_date is not in the aggregated CSV.
# The sessions table shows all sign-ups concentrated in Q1 2023.
# Insight is documented here for narrative completeness.
print("Sign-up distribution insight (from sessions table analysis):")
print("  ~5,200 of 5,998 customers signed up in Q1 2023 (a single quarter)")
print("  Near-zero organic growth before or after this spike")
print("  The user base was likely driven by a one-time campaign or partnership")
print("  This makes retention — not acquisition — the only viable lever")

---
## 3. EDA — Session Behaviour

Using aggregated session metrics from the feature table. Full session-level charts were produced from the raw sessions table in the original EDA and are in `reports/Final_TravelTide_BI_Analysis.pptx`.

In [ ]:
# ── Session duration vs. booking outcome proxy ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: avg session duration vs hotel spend (proxy for total booking value)
bookers = df[df['num_trips'] > 0]
axes[0].scatter(bookers['avg_session_duration'], bookers['money_spent_hotel'],
                alpha=0.3, s=15, color='steelblue')
axes[0].set_xlabel('Avg Session Duration (mins)')
axes[0].set_ylabel('Total Hotel Spend ($)')
axes[0].set_title('Session Duration vs. Spend\n(bookers only)')
axes[0].set_xlim(0, 300)
axes[0].axvline(10, color='tomato', linestyle='--', alpha=0.7, label='10-min mark')
axes[0].legend()

# Trip completion rate
trip_buckets = pd.cut(df['num_trips'], bins=[-1, 0, 2, 4, 100],
                      labels=['0 trips\n(no purchase)', '1–2 trips', '3–4 trips', '5+ trips'])
counts = trip_buckets.value_counts().sort_index()
axes[1].bar(counts.index, counts.values, color=['tomato','steelblue','steelblue','navy'])
axes[1].set_title('Customer Purchase Frequency Buckets')
axes[1].set_ylabel('Customers')
for i, (label, v) in enumerate(zip(counts.index, counts.values)):
    axes[1].text(i, v + 10, str(v), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

**Insight 6 — Fast converters spend the most.**  
The highest-value bookings (>$10,000) cluster in sessions under 10 minutes. Long sessions correlate with indecision, not intent. Reducing checkout friction for high-intent users is a higher-ROI intervention than extending session engagement.

**Insight 7 — 556 customers never booked (9.3%).**  
These signed up but made no purchases. They form a distinct non-buyer segment that should be treated separately from low-frequency buyers in the RFM model.

In [ ]:
# ── avg_bags as a segment signal ──────────────────────────────────────────
bags_by_trips = df.groupby('num_trips')['avg_bags'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(bags_by_trips['num_trips'], bags_by_trips['avg_bags'], color='steelblue')
ax.set_xlabel('Number of Trips')
ax.set_ylabel('Avg Checked Bags per Booking')
ax.set_title('Bag Usage Increases with Trip Frequency')
plt.tight_layout()
plt.show()

print("Customers with avg_bags > 1:", (df['avg_bags'] > 1).sum())
print("These are strong candidates for a free checked bag perk.")

---
## 4. RFM Feature Engineering

RFM measures customer value across three dimensions. We select features from the aggregated table that best approximate each dimension.

### Feature choices and rationale

| Dimension | Feature | Rationale |
|---|---|---|
| **Recency** | `time_after_booking` | Days between last booking and last session. Lower = more recently active. Non-bookers (0 trips) get value 0 but are handled as a separate flag — 0 here means "never booked", not "just booked". |
| **Frequency** | `num_trips` | Direct count of completed trips. More trips = higher frequency. |
| **Monetary** | `money_spent_hotel` | Total hotel spend. A partial proxy — flight spend is not captured here (see Limitations). No NULLs in this field. |

### Limitation note — Monetary
`money_spent_hotel` understates value for customers who fly but don't book hotels through TravelTide. A more complete metric would add `num_flights × avg_flight_fare`. This is a known gap; the current metric is used for consistency with the original segmentation.

In [ ]:
# ── Build RFM dataframe ─────────────────────────────────────────────────
rfm = df[['user_id']].copy()

# Recency: time_after_booking (lower = more recent = BETTER)
# Non-bookers have time_after_booking = 0, but so do some recent bookers.
# We distinguish them with a flag.
rfm['never_booked'] = df['num_trips'] == 0
rfm['recency_days'] = df['time_after_booking']

# Frequency
rfm['frequency'] = df['num_trips']

# Monetary — clip negatives to 0 (discount artefacts)
rfm['monetary'] = df['money_spent_hotel'].clip(lower=0)

print("RFM feature summary:")
print(rfm[['recency_days', 'frequency', 'monetary']].describe().round(2))
print(f"\nNever-booked customers: {rfm['never_booked'].sum()} ({100*rfm['never_booked'].mean():.1f}%)")

---
## 5. RFM Scoring

Each dimension is scored 1–3 using **quantile-based binning** (equal thirds of the population). This is a relative score — a score of 3 means "top third of our customers on this dimension."

**Key design decision — Recency scoring direction:**  
Lower `recency_days` = more recently active = **better** = score 3.  
We invert the label order in `pd.qcut` to achieve this.

**Key design decision — Never-bookers:**  
Customers with 0 trips are assigned `recency_score = 1`, `frequency_score = 1`, and `monetary_score = 1` directly, bypassing quantile scoring. Including them in the quantile calculation would distort scores for actual customers.

In [ ]:
# ── Score bookers (num_trips > 0) ─────────────────────────────────────────
bookers_mask = ~rfm['never_booked']
bookers = rfm[bookers_mask].copy()

# Recency: lower days = score 3 (invert with labels=[3,2,1])
bookers['recency_score'] = pd.qcut(
    bookers['recency_days'].rank(method='first'),
    q=3, labels=[3, 2, 1]   # 3 = lowest days = most recent
).astype(int)

# Frequency: higher trips = score 3
bookers['frequency_score'] = pd.qcut(
    bookers['frequency'].rank(method='first'),
    q=3, labels=[1, 2, 3]
).astype(int)

# Monetary: higher spend = score 3
bookers['monetary_score'] = pd.qcut(
    bookers['monetary'].rank(method='first'),
    q=3, labels=[1, 2, 3]
).astype(int)

# ── Assign never-bookers score 1 on all dimensions ────────────────────────
never_bookers = rfm[rfm['never_booked']].copy()
never_bookers[['recency_score', 'frequency_score', 'monetary_score']] = 1

# ── Combine ───────────────────────────────────────────────────────────────
rfm = pd.concat([bookers, never_bookers]).sort_values('user_id').reset_index(drop=True)
rfm['rfm_total'] = rfm['recency_score'] + rfm['frequency_score'] + rfm['monetary_score']

print("Score distributions:")
for col in ['recency_score', 'frequency_score', 'monetary_score']:
    print(f"  {col}: {rfm[col].value_counts().sort_index().to_dict()}")
print(f"\nRFM total range: {rfm['rfm_total'].min()} – {rfm['rfm_total'].max()}")

---
## 6. Segment Assignment

Segments are assigned using **Recency + Frequency** as the primary drivers. Monetary score is applied selectively to two segments — **At Risk** and **Promising** — where spend level materially changes the recommended strategy (a high-value at-risk customer needs more aggressive win-back spend than a low-value one).

**Design rationale:** For most segments, the perk recommendation is the same regardless of whether a customer spent $2k or $4k. The R+F combination is a better predictor of *when* to act; Monetary tells us *how much* to invest.

**Segment map:**

| R | F | M (where used) | Segment |
|---|---|---|---|
| 3 | 3 | — | Champions |
| 2–3 | 3 | — | Loyal Customers |
| 3 | 2 | — | Potential Champions |
| 1–2 | 3 | — | Do Not Lose |
| 1–2 | 2–3 | 3 | At Risk (High Value) |
| 1–2 | 2–3 | 1–2 | At Risk |
| 3 | 1 | 3 | Promising (High-Value) |
| 3 | 1 | 1–2 | Promising (Growth) |
| 2 | 2 | — | Need Attention |
| 2 | 1 | — | About to Sleep |
| 1 | 1 | — | Hibernating |
| any | any (never booked) | — | Hibernating |

In [ ]:
def assign_segment(row):
    r = row['recency_score']
    f = row['frequency_score']
    m = row['monetary_score']

    if row['never_booked']:
        return 'Hibernating'
    # --- High-value active ---
    if r == 3 and f == 3:
        return 'Champions'
    if f == 3 and r >= 2:
        return 'Loyal Customers'
    if r == 3 and f == 2:
        return 'Potential Champions'
    # --- Still frequent but cooling ---
    if f == 3 and r <= 2:
        return 'Do Not Lose'
    # --- Middle tier: needs attention before at-risk catch ---
    if r == 2 and f == 2:
        return 'Need Attention'
    # --- At risk: was frequent but recency has dropped ---
    if r <= 2 and f >= 2 and m == 3:
        return 'At Risk (High Value)'
    if r <= 2 and f >= 2:
        return 'At Risk'
    # --- New / low frequency ---
    if r == 3 and f == 1 and m == 3:
        return 'Promising (High-Value)'
    if r == 3 and f == 1:
        return 'Promising (Growth)'
    if r == 2 and f == 1:
        return 'About to Sleep'
    return 'Hibernating'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

# Summary
seg_summary = rfm.groupby('segment').agg(
    customers=('user_id', 'count'),
    avg_recency_days=('recency_days', 'mean'),
    avg_trips=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).sort_values('avg_monetary', ascending=False).round(1)

print(seg_summary.to_string())
print(f"\nTotal: {seg_summary['customers'].sum()} customers across {len(seg_summary)} segments")

---
## 7. Segment Analysis

### 7.1 Segment size and average LTV

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

seg_order = rfm.groupby('segment')['monetary'].mean().sort_values(ascending=False).index

# Bar: customer count per segment
counts = rfm['segment'].value_counts().reindex(seg_order)
axes[0].barh(seg_order[::-1], counts.values[::-1], color='steelblue')
axes[0].set_title('Customers per Segment')
axes[0].set_xlabel('Customers')
for i, v in enumerate(counts.values[::-1]):
    axes[0].text(v + 2, i, str(v), va='center', fontsize=9)

# Bar: avg LTV per segment
avg_ltv = rfm.groupby('segment')['monetary'].mean().reindex(seg_order)
axes[1].barh(seg_order[::-1], avg_ltv.values[::-1], color='tomato')
axes[1].set_title('Average LTV per Segment (Hotel Spend, USD)')
axes[1].set_xlabel('Average LTV ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
for i, v in enumerate(avg_ltv.values[::-1]):
    axes[1].text(v + 10, i, f'${v:,.0f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

### 7.2 The Value ≠ Frequency insight

The most frequent customers are not always the most valuable per unit of activity. This has real implications for where to invest retention spend.

In [ ]:
# Revenue per unit of frequency (how much each "trip" generates on average)
rev_per_freq = rfm.groupby('segment').apply(
    lambda g: g['monetary'].sum() / g['frequency'].replace(0, np.nan).sum()
).dropna().sort_values(ascending=False).round(0)

print("Revenue per trip by segment:")
print(rev_per_freq.apply(lambda x: f'${x:,.0f}').to_string())

# Highlight the key finding
ar_hv = rev_per_freq.get('At Risk (High Value)', 0)
loyal = rev_per_freq.get('Loyal Customers', 0)
print(f"\nAt Risk (High Value): ${ar_hv:,.0f} per trip")
print(f"Loyal Customers:      ${loyal:,.0f} per trip")
print(f"At Risk HV generates {ar_hv/loyal:.2f}x more revenue per trip than Loyal Customers.")
print("→ Retaining one At Risk HV customer protects more revenue than retaining one Loyal Customer.")

In [ ]:
# ── Revenue at risk: Do Not Lose ──────────────────────────────────────────
dnl = rfm[rfm['segment'] == 'Do Not Lose']
print(f"Do Not Lose segment:")
print(f"  Customers: {len(dnl):,}")
print(f"  Total LTV at risk: ${dnl['monetary'].sum():,.0f}")
print(f"  Avg recency: {dnl['recency_days'].mean():.1f} days since last booking")
print()
print("This is the second-largest revenue pool with declining recency.")
print("A targeted win-back campaign is cost-justified.")

---
## 8. Perk Recommendations

Perks are assigned based on what the segment's *data profile* suggests they value, not generic loyalty templates.

| Feature signal | Implied perk |
|---|---|
| High `avg_bags` | Free checked bag |
| Low recency, high monetary | Win-back discount |
| High frequency, low monetary | Frequency incentive (discount per trip) |
| High monetary, small segment | VIP / concierge treatment |
| Never booked / very low activity | Soft re-engagement only — minimal spend |

In [ ]:
# FIX 1: At Risk (HV) — 22% discount throughout (matches PPT script slide 22 and discount_pct column).
#         Original perk text incorrectly said 25% off.
# FIX 2: Promising (HV) — unified description: free cancellation + tiered benefits + early access.
#         PPT/exec summary said "loyalty rewards, tiered benefits, early access to sales";
#         user_perks.xlsx said "VIP light, free cancellation". Merged into one coherent offer.
# FIX 3: Potential Champions — frequency incentive framing retained (analytically grounded in
#         data: high num_trips, low money_spent_hotel). PPT called it "VIP program" which is
#         status language; this segment's problem is low spend per visit, not status.
# FIX 4: No hard LTV threshold stated here; PPT's "<$1,600" claim excluded (About to Sleep LTV
#         is $954 in notebook output, well under — no conflict. Threshold claim was in PPT script
#         only and used old Phase 1 numbers).
PERK_MAP = {
    'Champions':             'Loyalty rewards + quarterly exclusive offers + referral bonus',
    'Loyal Customers':       'Free checked bag + 10% loyalty discount + priority service',
    'Potential Champions':   'Frequency incentive: 10% off next 5 bookings + bundle deals + exclusive previews',
    'Do Not Lose':           'Win-back: 18% off next 3 bookings + free upgrade + personalised re-engagement',
    'At Risk (High Value)':  'Aggressive win-back: 22% off + lounge access + concierge outreach',  # was 25%, now 22%
    'At Risk':               'Basic re-engagement: 12% off + flash sale notification',
    'Promising (High-Value)':'Tiered loyalty benefits + early access to sales + free cancellation (2 bookings)',
    'Promising (Growth)':    'First-booking incentive: 12% off + free cancellation on first 2 bookings',
    'Need Attention':        'Engagement bonus: 12% off + free baggage + loyalty enrolment incentive',
    'About to Sleep':        'Re-engagement: 18% off + free hotel voucher + last-chance exclusive',
    'Hibernating':           'Soft re-engagement: 10% off + new feature notification (minimal investment)',
}

rfm['recommended_perk'] = rfm['segment'].map(PERK_MAP)

# Segment priority table
priority = {
    'At Risk (High Value)': 1, 'Do Not Lose': 1, 'Promising (High-Value)': 1,
    'Champions': 2, 'Loyal Customers': 2,
    'Potential Champions': 3, 'Need Attention': 3,
    'About to Sleep': 4, 'Promising (Growth)': 4,
    'Hibernating': 5, 'At Risk': 5
}
rfm['priority_tier'] = rfm['segment'].map(priority)

print("Final segment summary with perks:")
final = rfm.groupby('segment').agg(
    customers=('user_id', 'count'),
    avg_ltv=('monetary', 'mean'),
    priority=('priority_tier', 'first'),
    perk=('recommended_perk', 'first')
).sort_values('priority').round(0)
print(final[['customers','avg_ltv','priority','perk']].to_string())

---
## 8b. Perk Rationale — Grounded in Dataset Features

Each perk recommendation is justified by what the data actually shows about that segment. This section maps specific features from `customers.csv` to each recommended action.

**Signal features used:**

| Feature | Perk signal |
|---|---|
| `avg_bags > overall mean` | Free checked bag likely valued |
| `money_spent_hotel > overall mean` | Hotel-focused perks (upgrades, vouchers) relevant |
| `avg_km_flown > overall mean` | Long-haul travellers → lounge access worthwhile |
| `has_children` majority | Family bundle deals relevant |
| `avg_session_duration < overall mean` | Fast decision-makers → reduce checkout friction, not add steps |
| `num_trips` low + high recency | First/second booking incentives appropriate |
| `married` majority | Couple travel deals resonate |

In [ ]:
# ── Merge RFM segments back onto full feature table ───────────────────────
df_feat = df.merge(rfm[['user_id', 'segment']], on='user_id')

# Key features to profile per segment
FEAT_COLS = ['avg_bags', 'money_spent_hotel', 'avg_km_flown',
             'avg_session_duration', 'num_trips', 'num_flights',
             'married', 'has_children', 'age']

# Boolean columns → numeric
df_feat['married_num'] = df_feat['married'].astype(int)
df_feat['has_children_num'] = df_feat['has_children'].astype(int)

FEAT_COLS_PLOT = ['avg_bags', 'money_spent_hotel', 'avg_km_flown',
                  'avg_session_duration', 'num_trips',
                  'married_num', 'has_children_num', 'age']

# Overall means (baseline)
overall_means = df_feat[FEAT_COLS_PLOT].mean()
print("Overall feature means (baseline):")
print(overall_means.round(2).to_string())

In [ ]:
# ── Per-segment feature profile (ratio vs. overall mean) ─────────────────
seg_profiles = df_feat.groupby('segment')[FEAT_COLS_PLOT].mean()

# Normalise: ratio to overall mean (1.0 = average, 1.2 = 20% above, 0.8 = 20% below)
seg_profiles_norm = seg_profiles.div(overall_means)

# Order segments by priority
seg_order = ['At Risk (High Value)', 'Do Not Lose', 'Promising (High-Value)',
             'Champions', 'Loyal Customers', 'Potential Champions', 'Need Attention',
             'About to Sleep', 'Promising (Growth)', 'At Risk', 'Hibernating']
seg_profiles_norm = seg_profiles_norm.reindex(seg_order)

print("Segment feature profiles (ratio vs. overall mean):")
print(seg_profiles_norm.round(2).to_string())

In [ ]:
# ── Heatmap: feature signals by segment ──────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

fig, ax = plt.subplots(figsize=(14, 7))

col_labels = {
    'avg_bags': 'Avg Bags',
    'money_spent_hotel': 'Hotel Spend',
    'avg_km_flown': 'Avg KM Flown',
    'avg_session_duration': 'Session Duration',
    'num_trips': 'Num Trips',
    'married_num': '% Married',
    'has_children_num': '% Has Children',
    'age': 'Avg Age'
}

data = seg_profiles_norm.values
im = ax.imshow(data, cmap='RdYlGn', vmin=0.5, vmax=1.5, aspect='auto')

ax.set_xticks(range(len(FEAT_COLS_PLOT)))
ax.set_xticklabels([col_labels[c] for c in FEAT_COLS_PLOT], rotation=35, ha='right', fontsize=10)
ax.set_yticks(range(len(seg_order)))
ax.set_yticklabels(seg_order, fontsize=10)

# Annotate cells
for i in range(len(seg_order)):
    for j in range(len(FEAT_COLS_PLOT)):
        val = data[i, j]
        color = 'white' if abs(val - 1.0) > 0.3 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='Ratio vs. overall mean (1.0 = average)')
ax.set_title('Segment Feature Profiles\n(green = above average, red = below average)', fontsize=12, pad=15)
plt.tight_layout()
plt.show()

print("\nReading guide: 1.20 = 20% above overall mean for that feature.")
print("Values above 1.15 (strong green) or below 0.85 (strong red) are meaningful signals.")

In [ ]:
# ── Perk justification table: which features drive each perk ─────────────
PERK_JUSTIFICATION = {
    'Champions': {
        'perk': 'Loyalty rewards + quarterly exclusives + referral bonus',
        'signals': ['num_trips ↑↑ (highest frequency)', 'money_spent_hotel ↑↑', 'avg_km_flown ↑'],
        'rationale': 'These customers already exhibit the ideal behaviour. Reward loyalty to protect it — '
                     'not to change it. High trip count and hotel spend justify exclusive access perks. '
                     'Referral bonus leverages their engagement to drive acquisition.'
    },
    'Loyal Customers': {
        'perk': 'Free checked bag + 10% loyalty discount + priority service',
        'signals': ['avg_bags ↑ (highest of all segments)', 'num_trips ↑↑', 'money_spent_hotel ↑↑'],
        'rationale': 'Loyal Customers have the highest avg_bags of any segment — a free checked bag '
                     'directly addresses a pain point they demonstrably have. High trip count and spend '
                     'justify a loyalty discount without over-discounting.'
    },
    'Potential Champions': {
        'perk': 'Frequency incentive: 10% off next 5 bookings + bundle deals + exclusive previews',
        'signals': ['num_trips ↑ (high)', 'money_spent_hotel ↓ (below avg)', 'avg_session_duration ↑'],
        'rationale': 'High visit frequency but below-average hotel spend signals they book flights '
                     'but skip hotel add-ons. Bundle deals directly target this gap. The 10% frequency '
                     'incentive rewards the behaviour they already show (visiting) to convert it into spend.'
    },
    'Do Not Lose': {
        'perk': 'Win-back: 18% off next 3 bookings + free upgrade + personalised re-engagement',
        'signals': ['num_trips ↑↑ (high)', 'money_spent_hotel ↑↑ (high)', 'recency ↓ (declining)'],
        'rationale': 'Profile matches Champions/Loyal in spend and frequency — the only difference is '
                     'recency. These customers know the product and used to value it. A time-limited '
                     'win-back offer (not a permanent discount) is the right lever. Free upgrade signals '
                     'valued status without a permanent cost.'
    },
    'At Risk (High Value)': {
        'perk': 'Aggressive win-back: 22% off + lounge access + concierge outreach',
        'signals': ['avg_km_flown ↑↑ (highest of any segment)', 'money_spent_hotel ↑↑', 'recency ↓↓'],
        'rationale': 'Highest avg_km_flown of any segment — these are long-haul travellers. Lounge access '
                     'is a tangible, high-perceived-value perk for this profile. 22% discount is aggressive '
                     'because revenue per trip ($1,503) is the second highest — retention ROI is strong. '
                     'Concierge outreach is warranted given their LTV ($4,187 avg).'
    },
    'Promising (High-Value)': {
        'perk': 'Tiered loyalty benefits + early access to sales + free cancellation (2 bookings)',
        'signals': ['money_spent_hotel ↑↑ (highest avg LTV)', 'num_trips ↓ (low frequency)', 'recency ↑ (recent)'],
        'rationale': 'High hotel spend despite low trip count confirms high-ticket bookings (likely luxury '
                     'or long-stay). Free cancellation removes the key barrier for infrequent high-value '
                     'bookers — they may hesitate to commit. Tiered loyalty entry point nudges them toward '
                     'repeat behaviour.'
    },
    'Need Attention': {
        'perk': 'Engagement bonus: 12% off + free baggage + loyalty enrolment incentive',
        'signals': ['num_trips mid', 'money_spent_hotel mid', 'avg_bags mid'],
        'rationale': 'The most "average" segment across all features — centre of the funnel. A modest '
                     'discount combined with a tangible perk (free baggage) provides a nudge without '
                     'over-investing. Loyalty enrolment incentive attempts to formalise their relationship '
                     'with the platform before they drift downward.'
    },
    'About to Sleep': {
        'perk': 'Re-engagement: 18% off + free hotel voucher + last-chance exclusive',
        'signals': ['num_trips ↓ (low, ~1-2)', 'money_spent_hotel ↓', 'recency mid (not yet cold)'],
        'rationale': 'Low trip count and spend but not yet cold. The free hotel voucher provides a '
                     'concrete reason to book — more tangible than a discount alone. Last-chance framing '
                     'creates urgency appropriate for their trajectory.'
    },
    'Promising (Growth)': {
        'perk': 'First-booking incentive: 12% off + free cancellation on first 2 bookings',
        'signals': ['num_trips ↓ (1-2 trips or none)', 'recency ↑ (recent sessions)', 'money_spent_hotel ↓'],
        'rationale': 'Recent but low frequency — classic early-stage customer. Free cancellation on '
                     'first 2 bookings removes commitment anxiety for new bookers. 12% off is '
                     'proportionate to low current LTV.'
    },
    'At Risk': {
        'perk': 'Basic re-engagement: 12% off + flash sale notification',
        'signals': ['num_trips mid-low', 'money_spent_hotel ↓ (low)', 'recency ↓'],
        'rationale': 'Mid-frequency but low spend and declining recency. Revenue per trip ($345) is '
                     'the lowest of any segment — aggressive retention spend is not justified. Minimal '
                     'investment: flash sale notification keeps the door open at near-zero cost.'
    },
    'Hibernating': {
        'perk': 'Soft re-engagement: 10% off + new feature notification',
        'signals': ['num_trips ↓↓ (lowest)', 'money_spent_hotel ↓↓', 'recency ↓↓'],
        'rationale': 'Lowest activity across all three RFM dimensions. 556 of these never booked at all. '
                     'Hard win-back investment is not cost-justified. A low-cost notification keeps the '
                     'brand visible; if they re-engage, they move segments naturally.'
    },
}

print("=" * 80)
print("PERK RATIONALE — GROUNDED IN DATASET FEATURES")
print("=" * 80)
for seg in seg_order:
    j = PERK_JUSTIFICATION.get(seg, {})
    print(f"\n{'─'*60}")
    print(f"SEGMENT: {seg}")
    print(f"  Perk:     {j.get('perk', 'N/A')}")
    print(f"  Signals:  {' | '.join(j.get('signals', []))}")
    print(f"  Why:      {j.get('rationale', '')}")
print("\n" + "=" * 80)

In [ ]:
# ── Key feature deltas: which features are most distinctive per segment ──
print("Top 2 most distinctive features per segment (largest deviation from mean):\n")
for seg in seg_order:
    row = seg_profiles_norm.loc[seg]
    # Rank by absolute deviation from 1.0
    deviations = (row - 1.0).abs().sort_values(ascending=False)
    top2 = deviations.head(2).index.tolist()
    vals = [f"{c} ({seg_profiles_norm.loc[seg, c]:.2f}x)" for c in top2]
    print(f"  {seg:<28} → {', '.join(vals)}")
print("\n(Values >1.0 = above average for that segment; <1.0 = below average)")

---
## 9. Export

Outputs are written to `data/processed/`. Both the full customer-level file and the segment summary are saved.

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

# Full customer-level RFM output
output_cols = ['user_id', 'recency_days', 'frequency', 'monetary',
               'recency_score', 'frequency_score', 'monetary_score',
               'rfm_total', 'never_booked', 'segment', 'recommended_perk', 'priority_tier']
rfm[output_cols].to_csv('data/processed/rfm_corrected.csv', index=False)
print(f"Saved: data/processed/rfm_corrected.csv ({len(rfm):,} rows)")

# Segment summary
seg_export = rfm.groupby('segment').agg(
    customer_count=('user_id', 'count'),
    avg_ltv=('monetary', 'mean'),
    avg_recency_days=('recency_days', 'mean'),
    avg_trips=('frequency', 'mean'),
    priority_tier=('priority_tier', 'first'),
    recommended_perk=('recommended_perk', 'first')
).sort_values('priority_tier').round(2)

seg_export.to_csv('data/processed/segment_summary.csv')
print(f"Saved: data/processed/segment_summary.csv ({len(seg_export)} segments)")

print("\nDone. See CORRECTIONS.md for the audit trail of changes from original notebook.")